# PreProcessing du csv supervise

## Initialisation

In [114]:
import pandas as pd
import cv2
import numpy as np
import json
import os

In [115]:
df = pd.read_csv('../../kidneyData.csv')

##### Pour etre conforme a notre version nous allons retirer `/content/data` du CSV

In [116]:
df['path'] = df['path'].str.replace('/content/data/', '')
print(df['path'])

0        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
1        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
2        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
3        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
4        CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
                               ...                        
12441    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12442    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12443    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12444    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
12445    CT KIDNEY DATASET Normal, CYST, TUMOR and STON...
Name: path, Length: 12446, dtype: str


## Suppression des column inutile

In [117]:
df = df.iloc[:, 1:]
df = df.drop('path', axis=1)
df = df.drop('Class', axis=1)
print(df)

            image_id   diag  target
0      Tumor- (1044)  Tumor       3
1        Tumor- (83)  Tumor       3
2       Tumor- (580)  Tumor       3
3      Tumor- (1701)  Tumor       3
4      Tumor- (1220)  Tumor       3
...              ...    ...     ...
12441   Cyst- (2522)   Cyst       0
12442   Cyst- (2627)   Cyst       0
12443    Cyst- (972)   Cyst       0
12444   Cyst- (2323)   Cyst       0
12445   Cyst- (2145)   Cyst       0

[12446 rows x 3 columns]


## Ajout des array dans la column matrice 

Pour faire cela, nous allons passer dans les 4 dossiers et dans chaque image des dossiers.
On va appliquer une redimension de 128 par 128 tout en garddant les proportions.
Nous allons egalement gerer le probleme des couleurs et basculer toute les image en gris.
Une fois cela fait on ajoute l'array numpy de la photo directement dans la ligne correspondante dans l'ordre

In [118]:
df = pd.read_csv("../../Supervise.csv")

taille_standard = (128, 128)

dossiers = {
    "Tumor": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Tumor/",
    "Normal": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Normal/",
    "Stone": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Stone/",
    "Cyst": "../../CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/Cyst/"
}

toutes_les_matrices = []

for _, row in df.iterrows():

    dossier = dossiers[row["diag"]]

    nom_fichier = row["image_id"] + ".jpg"

    chemin_complet = os.path.join(dossier, nom_fichier)

    img = cv2.imread(chemin_complet)

    if img is None:
        print("Erreur :", chemin_complet)
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    img_resized = cv2.resize(gray, taille_standard)

    toutes_les_matrices.append(img_resized.flatten())

X = np.array(toutes_les_matrices)

np.save("../../Matrice.npy", X)

print("X shape :", X.shape)
print("CSV :", len(df))

X shape : (12446, 16384)
CSV : 12446


##### JUstification de l'utilisation du module CV2 dans le fichier `JustificationCV2` a la racine du dossier actuel

## Export de notre dataset en CSV

In [119]:
df.to_csv("../../Supervise.csv")